## Общие настройки симуляции

In [1]:
import time
import jax
import jax.numpy as jnp
import pytreeclass as tc
from loguru import logger
import fdtdx
import optax
import chex
%matplotlib inline

backward = False
exp_logger = fdtdx.Logger(experiment_name="TopOpt_Si_model_P=900_R=0.4_phi=0.5pi",) #как будет назваться папка
seed = 42
key = jax.random.PRNGKey(seed=seed)
logger.info(f"Simulation seed: {seed}")

wavelength = 1.55e-6  
frequency = fdtdx.constants.c / wavelength  
period = fdtdx.constants.wavelength_to_period(wavelength)  
    
logger.info(f"Wavelength: {wavelength*1e9:.1f} nm") 
logger.info(f"Frequency: {frequency*1e-12:.1f} THz")
logger.info(f"Period: {period*1e15:.1f} fs")

config = fdtdx.SimulationConfig(
        time=100e-15,           
        resolution=20e-9,      
        dtype=jnp.float32,     
        courant_factor=0.99, )  
    
period_steps = round(period / config.time_step_duration)
all_time_steps = list(range(config.time_steps_total))
    
logger.info(f"Total time steps: {config.time_steps_total}")
logger.info(f"Time step duration: {config.time_step_duration*1e18:.1f} as")
logger.info(f"Steps per optical period: {period_steps}")
logger.info(f"Max travel distance: {config.max_travel_distance*1e6:.1f} μm")
    
gradient_config = fdtdx.GradientConfig(
recorder=fdtdx.Recorder(
            modules=[
                
                fdtdx.LinearReconstructEveryK(k=5),
                fdtdx.DtypeConversion(dtype=jnp.bfloat16),
            ]
        )
    )
config = config.aset("gradient_config", gradient_config) 
    

constraints = [] 
    
volume = fdtdx.SimulationVolume(
        partial_real_shape=(0.9e-6, 0.9e-6, 5.2e-6),  
        material=fdtdx.Material(  
            permittivity=fdtdx.constants.relative_permittivity_air,
            permeability=1.0,  
        )
)

bound_cfg =fdtdx.BoundaryConfig(boundary_type_minx='periodic',
                                boundary_type_maxx='periodic', 
                                boundary_type_miny='periodic', 
                                boundary_type_maxy='periodic',
                                boundary_type_minz='pml', 
                                boundary_type_maxz='pml', 
                                thickness_grid_minx=10, 
                                thickness_grid_maxx=10, 
                                thickness_grid_miny=10, 
                                thickness_grid_maxy=10, 
                                thickness_grid_minz=10,
                                thickness_grid_maxz=10,
)
    
# создание объектов границ
bound_dict, boundary_constraints = fdtdx.boundary_objects_from_config(bound_cfg, volume) 
constraints.extend(boundary_constraints)

/usr/local/lib/python3.11/dist-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

19.03.2026 00:05:05.042 | /home/workspace/fdtdx/src/fdtdx/utils/logger.py:133 - Starting experiment 
TopOpt_Si_model_P=900_R=0.4_phi=0.5pi in 
/home/workspace/outputs/nobackup/2026-03-19/TopOpt_Si_model_P=900_R=0.4_phi=0.5pi/00-05-05.023918

19.03.2026 00:05:05.335 | /tmp/ipykernel_159284/3791503899.py:15 - Simulation seed: 42

19.03.2026 00:05:05.337 | /tmp/ipykernel_159284/3791503899.py:21 - Wavelength: 1550.0 nm

19.03.2026 00:05:05.338 | /tmp/ipykernel_159284/3791503899.py:22 - Frequency: 193.4 THz

19.03.2026 00:05:05.338 | /tmp/ipykernel_159284/3791503899.py:23 - Period: 5.2 fs

19.03.2026 00:05:05.339 | /tmp/ipykernel_159284/3791503899.py:34 - Total time steps: 2623

19.03.2026 00:05:05.340 | /tmp/ipykernel_159284/3791503899.py:35 - Time step duration: 38.1 as

19.03.2026 00:05:05.341 | /tmp/ipykernel_159284/3791503899.py:36 - Steps per optical period: 136

19.03.2026 00:05:05.342 | /tmp/ipykernel_159284/3791503899.py:37 - Max travel distance: 30.0 μm

## Настройка области FDTD

In [2]:
from fdtdx.core.jax.pytrees import autoinit, frozen_private_field
from fdtdx.objects.device.parameters.transform import SameShapeTypeParameterTransform
import jax.numpy as jnp

@autoinit
class PeriodicEdges2D(SameShapeTypeParameterTransform):
    _all_arrays_2d: bool = frozen_private_field(default=True)

    def __call__(self, params: dict[str, jnp.ndarray], **kwargs) -> dict[str, jnp.ndarray]:
        del kwargs
        result = {}
        for k, v in params.items():
            vertical_axis = v.shape.index(1)
            v_2d = v.squeeze(vertical_axis)          

             
            avg_x = 0.5 * (v_2d[:, 0] + v_2d[:, -1])
            v_2d = v_2d.at[:, 0].set(avg_x)
            v_2d = v_2d.at[:, -1].set(avg_x)

            avg_y = 0.5 * (v_2d[0, :] + v_2d[-1, :])
            v_2d = v_2d.at[0, :].set(avg_y)
            v_2d = v_2d.at[-1, :].set(avg_y)

            result[k] = jnp.expand_dims(v_2d, vertical_axis)
        return result

In [3]:
materials_config = {
        "Air": fdtdx.Material(permittivity=fdtdx.constants.relative_permittivity_air),
        "Silicon": fdtdx.Material(permittivity=12.0852),
    }
# подложка
substrate = fdtdx.UniformMaterialObject( 
    name="Silicon_Substrate", #тут имя
    partial_real_shape=(None, None, 0.5e-6),  
    material=fdtdx.Material(
        permittivity=12.0852),
    color=fdtdx.colors.GREY,)
constraints.append(
    substrate.place_relative_to(
        volume,
        axes=2,                
        own_positions=-1,     
        other_positions=-1,    
    )
)

silica = fdtdx.UniformMaterialObject( 
    name="SiO2_Layer", #тут имя
    partial_real_shape=(None, None, 2e-6),  
    material=fdtdx.Material(permittivity=2.08514))
                
constraints.extend([ 
        silica.place_at_center(volume, axes=(0, 1)),  # центрировать по X и Y
        silica.place_above(substrate),                 # разместить над подложкой
    ])


# оптимизаируемые девайсы со сложной геометрией
height = 220e-9
voxel_size = 20e-9
     
device = fdtdx.Device(
    name="Device",
    partial_real_shape=(0.9e-6, 0.9e-6, height),
    materials=materials_config,
    param_transforms=[
        PeriodicEdges2D(),
        #DiagonalSymmetry2D(min_min_to_max_max=False),
        fdtdx.GaussianSmoothing2D(std_discrete=3), #параметры сглаживания для топологической оптимизации
        fdtdx.SubpixelSmoothedProjection(),
        fdtdx.TanhProjection(),
         ],
    partial_voxel_real_shape=(voxel_size, voxel_size, height),
)
constraints.extend([
    device.place_at_center(volume, axes=(0, 1)),  # центрировать по X и Y
    device.place_relative_to(
        silica,
        axes=(2),
        own_positions=(-1), 
        other_positions=(1),
        #grid_margins=(-bound_cfg.thickness_grid_maxx, bound_cfg.thickness_grid_miny, 0), #отступ по сетке
        #margins=(-0.2e-6, 0.2e-6, 0), #сколько отступ
    )
])

plane_sourse = fdtdx.UniformPlaneSource(
    name="Plane_Sourse",
    partial_real_shape=(0.9e-6, 0.9e-6, None),
    partial_grid_shape=(None, None, 1),
    fixed_E_polarization_vector=(1, 0, 0),
    wave_character=fdtdx.WaveCharacter(wavelength=wavelength),
    direction="-",    
    azimuth_angle=0.0,               #азимутальный угол (градусы)
    elevation_angle=0.0,             #угол возвышения (градусы)

    temporal_profile=fdtdx.SingleFrequencyProfile(
    phase_shift=0.0,)
    #temporal_profile=fdtdx.GaussianPulseProfile(
    #    spectral_width=43e12,               # ширина спектра (Гц)
    #    center_frequency=frequency,        # центральная частота
    #),
)
constraints.extend(
    [
        plane_sourse.place_at_center(volume, axes=(0, 1)),  # центрировать по X и Y
        plane_sourse.place_relative_to(
            device,
            #circle,
            axes=(2),
            own_positions=(-1),
            other_positions=(1),
            margins=(1.2e-6),
        ),
    ]
)



output_power_monitor = fdtdx.PoyntingFluxDetector(
    name="Output_Power",
    partial_grid_shape=(None, None, 1),
    partial_real_shape=(0.9e-6, 0.9e-6, None),
#вся плоскость yz
    direction="+",                          # направление измерения
    reduce_volume=True,                     # суммировать по всей поверхности   
    switch=fdtdx.OnOffSwitch(fixed_on_time_steps = all_time_steps[-period_steps:]),
    exact_interpolation=True,
    color=fdtdx.colors.MAGENTA,
)
constraints.extend(
    [
        output_power_monitor.place_at_center(volume, axes=(0, 1)),  # центрировать по X и Y
        output_power_monitor.place_relative_to(
            plane_sourse,
            axes=(2),
            own_positions=(-1),
            other_positions=(1),
            margins=(0.5e-6),
        ),
    ]
)


input_power_monitor = fdtdx.PoyntingFluxDetector(
    name="Input_Power",
    partial_grid_shape=(None, None, 1),
    partial_real_shape=(0.9e-6, 0.9e-6, None),
    direction="-",                          
    reduce_volume=True,                     
    switch=fdtdx.OnOffSwitch(fixed_on_time_steps = all_time_steps[4*period_steps : 5*period_steps]),
    exact_interpolation=True,
    color=fdtdx.colors.GREEN,
)

constraints.extend(
    [
        input_power_monitor.place_at_center(volume, axes=(0, 1)),  
        input_power_monitor.place_relative_to(
            plane_sourse,
            axes=(2),
            own_positions=(-1),
            other_positions=(1),
            grid_margins = -3
            #margins=(-1.18e-6),
        ),
    ]
)


phasor_detector = fdtdx.PhasorDetector(
        name="Frequency_Analysis",
        wave_characters=[
            fdtdx.WaveCharacter(wavelength=1.55e-6),
        ],
        components=("Ex", "Ey", "Ez"),
        reduce_volume=False,
        dtype=jnp.complex64,
        plot=False,  
        switch=fdtdx.OnOffSwitch(fixed_on_time_steps=all_time_steps[-period_steps:]),
    )
constraints.extend(phasor_detector.same_position_and_size(output_power_monitor))

boundary_objects = list(bound_dict.values())

all_objects = [
    volume,
    substrate,
    silica,
    device,
    plane_sourse,
    output_power_monitor,
    input_power_monitor,
    phasor_detector,
    *boundary_objects,  # ← добавили границы
]

key, subkey = jax.random.split(key)

objects, arrays, params, config, info = fdtdx.place_objects(
    object_list=all_objects,
    config=config,
    constraints=constraints,
    key=key,
)
'''
from make_initial_distribution import make_initial_distribution

init_params = make_initial_distribution(
    device_shape=params[device.name].shape,
    voxel_size=20e-9,
    radius=260e-9,
    disk_density=0.5,
    background_density=0.55,
    save_path="initial_params.npy",
)

params[device.name] = init_params
#np.savez("/home/workspace/disk_distribution", init_params)
'''
#saved_params = jnp.load("outputs/nobackup/2026-03-18/TopOpt_Si_model_P=900_R=0.4_phi=0.5pi/18-32-36.138207/device/params/params_510_Device.npy")
#params[device.name] = saved_params


'\nfrom make_initial_distribution import make_initial_distribution\n\ninit_params = make_initial_distribution(\n    device_shape=params[device.name].shape,\n    voxel_size=20e-9,\n    radius=260e-9,\n    disk_density=0.5,\n    background_density=0.55,\n    save_path="initial_params.npy",\n)\n\nparams[device.name] = init_params\n#np.savez("/home/workspace/disk_distribution", init_params)\n'

## Оптимизация 

In [4]:
start_idx = 0
epochs = 201

R0 = 0.8
phi0 = jnp.pi
A1 = 500
evaluation = False


if not evaluation:
    schedule_finetune: optax.Schedule = optax.warmup_cosine_decay_schedule(
        init_value=1e-5,
        peak_value=0.005,
        end_value=0.0005,
        warmup_steps=10,#10
        decay_steps=round(0.9 * epochs),
    )
    optimizer_finetune = optax.inject_hyperparams(optax.nadam)(learning_rate=schedule_finetune)
    #optimizer_finetune = optax.chain(
    #optax.clip_by_global_norm(1.0),
    #optax.inject_hyperparams(optax.nadam)(learning_rate=schedule_finetune),
    

    optimizer_finetune = optax.MultiSteps(optimizer_finetune, every_k_schedule=1)
    opt_state_finetune: optax.OptState = optimizer_finetune.init(params)
    
def custom_schedule(idx: chex.Numeric) -> chex.Numeric:
    beta_schedule = optax.linear_schedule(1, 50, epochs)
    return jax.lax.cond(
        idx < epochs - 2,
        lambda: beta_schedule(idx),
        lambda: jnp.inf
    )


    
exp_logger.savefig(
    exp_logger.cwd,
    "setup.png",
    fdtdx.plot_setup(
        config=config,
        objects=objects,
        #exclude_object_list=exclude_object_list,
    ),
)

changed_voxels = exp_logger.log_params(
    iter_idx=-1,
    params=params,
    objects=objects,
    export_stl=True,
    export_figure=True,
    beta=custom_schedule(start_idx),
)
    
x, tmp, _ = fdtdx.apply_params(arrays, objects, params, key, beta=custom_schedule(start_idx))

In [5]:
def loss_func(
    params: fdtdx.ParameterContainer,
    arrays: fdtdx.ArrayContainer,
    key: jax.Array,
    idx: int,
):
    arrays, new_objects, info = fdtdx.apply_params(arrays, objects, params, key, beta=custom_schedule(idx))

    final_state = fdtdx.run_fdtd(
        arrays=arrays,
        objects=new_objects,
        config=config,
        key=key,
    )

    _, arrays = final_state

    #достать данные из монитора
    input_power = arrays.detector_states[input_power_monitor.name]["poynting_flux"].sum()
    output_power = arrays.detector_states[output_power_monitor.name]["poynting_flux"].sum()
    R = output_power / jnp.maximum(input_power, 1e-10)

    
    phasor = arrays.detector_states[phasor_detector.name]["phasor"]
    
    #E_near = phasor[0, 0, 0, :, :, 0] 
    '''

    E_orders = jnp.fft.fftshift(jnp.fft.fft2(E_near))
    cx, cy = E_near.shape[0] // 2, E_near.shape[1] // 2
    E_order_00 = E_orders[cx, cy]

    
    z_struct_to_det = 2.85e-6  
    k0 = 2 * jnp.pi / wavelength
    n_medium = 1.0

    
    avg_phasor = E_order_00 * jnp.exp(-1j * k0 * n_medium * z_struct_to_det)
    '''
    avg_phasor = phasor[0, 0, 0].mean()  

    # Комплексный коэфф. отражения
    r_amp = jnp.sqrt(jnp.maximum(R, 1e-20))
    r0_amp = jnp.sqrt(R0)

    abs_sq = avg_phasor.real**2 + avg_phasor.imag**2  # |z|² без sqrt
    abs_val = jnp.sqrt(jnp.maximum(abs_sq, 1e-30))     # |z| с защитой внутри sqrt
    avg_phasor_norm = avg_phasor / abs_val           #  нормировка

    target = jnp.exp(1j * phi0)

    r_complex = r_amp * avg_phasor_norm
    r0_complex = r0_amp * target


    #FoM = jnp.abs(r_complex - r0_complex)**2

    diff = r_complex - r0_complex
    FoM = diff.real**2 + diff.imag**2

    phi = jnp.angle(avg_phasor)  # только для логирования, не в FoM

    objective = A1 * FoM


    
    if evaluation and backward:
        _, arrays = fdtdx.full_backward(
            state=final_state,
            objects=new_objects,
            config=config,
            key=key,
            record_detectors=True,
            reset_fields=True,
        )

        
    new_info = {
            "input_power": input_power,
            "output_power": output_power,
            "reflection_coefficient": R,
            "phase": phi,
            "FoM": objective,
        
            **info,
    }
    return objective, (arrays, new_info) 

compile_start_time = time.time()
jit_task_id = exp_logger.progress.add_task("JIT", total=None)
idx_dummy_arr = jnp.asarray(start_idx, dtype=jnp.float32)
if evaluation:
    jitted_loss = jax.jit(loss_func, donate_argnames=["arrays"]).lower(params, arrays, key, idx_dummy_arr).compile()
else:
    jitted_loss = (
        jax.jit(jax.value_and_grad(loss_func, has_aux=True), donate_argnames=["arrays"])
        .lower(params, arrays, key, idx_dummy_arr)
        .compile()
    )
compile_delta_time = time.time() - compile_start_time
exp_logger.progress.update(jit_task_id, total=1, completed=1, refresh=True)

optim_task_id = exp_logger.progress.add_task("Optimization", total=1 if evaluation else epochs)
for epoch in range(start_idx, start_idx + 1 if evaluation else epochs):

    run_start_time = time.time()
    key, subkey = jax.random.split(key)
    idx_arr = jnp.asarray(epoch, dtype=jnp.float32)
    if evaluation:
        loss, (arrays, info) = jitted_loss(params, arrays, subkey, idx_arr)
    else:
        (loss, (arrays, info)), grads = jitted_loss(params, arrays, subkey, idx_arr)

        updates, opt_state_finetune = optimizer_finetune.update(grads, opt_state_finetune, params) # type: ignore
        info["lr"] = opt_state_finetune.inner_opt_state.hyperparams["learning_rate"]
        
        #updates, opt_state_finetune = optimizer_finetune.update(grads, opt_state_finetune, params)
        #inner_state, _ = opt_state_finetune
        #info["lr"] = inner_state.hyperparams["learning_rate"]
        #info["lr"] = schedule_finetune(epoch)
        params = optax.apply_updates(params, updates)
        params = jax.tree_util.tree_map(lambda p: jnp.clip(p, 0, 1), params)
        info["grad_norm"] = optax.global_norm(grads)
        info["update_norm"] = optax.global_norm(updates)

        # tmp[overlap_detector.name].compute_overlap(arrays.detector_states[overlap_detector.name])
    jax.block_until_ready((arrays, info))
    runtime_delta = time.time() - run_start_time
    info["runtime"] = runtime_delta
    info["attenuation"] = 10 * jnp.log10(jnp.maximum(loss, 1e-10))

    if evaluation:
        logger.info(f"{compile_delta_time=}")
        logger.info(f"{runtime_delta=}")

        # log parameters
    changed_voxels = exp_logger.log_params(
        iter_idx=epoch,
        params=params,
        objects=objects,
        export_stl=True,
        export_figure=True,
        beta=custom_schedule(epoch)
    )
    info["changed_voxels"] = changed_voxels

    #logger.info(f"Время компиляции: {compile_delta_time:.2f} с")
    #logger.info(f"Время выполнения: {runtime_delta:.2f} с")
    #logger.info(f"Входящая мощность: {info['input_power']:.4f}")
    #logger.info(f"Отраженная мощность: {info['output_power']:.4f}")
    logger.info(f"Коэффициент отражения: {info['reflection_coefficient']:.4f}")
    logger.info(f"Фаза: {info['phase']:.4f}")
    logger.info(f"FoM: {info['FoM']:.4f}")
    logger.info(f"grad_norm={info['grad_norm']}, update_norm={info['update_norm']}")


        # videos
    exp_logger.log_detectors(iter_idx=epoch, objects=objects, detector_states=arrays.detector_states)

    exp_logger.write(info)
    exp_logger.progress.update(optim_task_id, advance=1)

19.03.2026 00:06:22.516 | /tmp/ipykernel_159284/3413559665.py:168 - Коэффициент отражения: 0.2635

19.03.2026 00:06:22.519 | /tmp/ipykernel_159284/3413559665.py:169 - Фаза: 2.9813

19.03.2026 00:06:22.520 | /tmp/ipykernel_159284/3413559665.py:170 - FoM: 11.2606

19.03.2026 00:06:22.521 | /tmp/ipykernel_159284/3413559665.py:171 - grad_norm=0.8953965902328491, 
update_norm=0.0006631625583395362

/usr/local/lib/python3.11/dist-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

19.03.2026 00:30:08.449 | /tmp/ipykernel_159284/3413559665.py:170 - FoM: 0.2498

19.03.2026 00:30:08.450 | /tmp/ipykernel_159284/3413559665.py:171 - grad_norm=0.0, update_norm=0.000136563103296794

/usr/local/lib/python3.11/dist-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/usr/local/lib/python3.11/dist-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

 FoM  0.…  in…  62…  out…  23…  pha…  -3…  ref…  0.…  lr  0.0…  gr…  0.0  upd…  0.…  run…  5.…  att…  -6…  cha…  0 